In [ ]:
# ==========================================
# Cell 1: 基础配置 (V3.2.0 - Global + Kalman + Sliding Window)
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data' 

OUT_DIR = ROOT / 'output_320'
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'

for d in [CM_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Base Dir: {OUT_DIR}')

In [ ]:
# ==========================================
# Cell 2: 数据读取与 Kalman Filter
# ==========================================
stations_list_path = DATA_DIR / 'stations_list'
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
TOTAL_STATIONS = len(station_ids)

def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')

def apply_kalman_filter(curve, process_variance=1e-4, measurement_variance=1e-2):
    n = len(curve)
    if n == 0: return curve
    xhat, P = np.zeros(n), np.zeros(n)
    xhat[0], P[0] = curve[0], 1.0
    for k in range(1, n):
        xhat_minus = xhat[k-1]
        P_minus = P[k-1] + process_variance
        K = P_minus / (P_minus + measurement_variance) 
        xhat[k] = xhat_minus + K * (curve[k] - xhat_minus)
        P[k] = (1 - K) * P_minus
    return xhat

In [ ]:
# ==========================================
# Cell 3: 滑动窗口数据增强 Dataset
# ==========================================
def process_and_slice_curve(curve_144, is_train, window_size=132, stride=2):
    """
    1. 先对完整的 144 长度序列进行填充和卡尔曼滤波
    2. 根据是否是训练集，进行滑动窗口裁剪或中心裁剪
    3. 对裁剪后的序列提取统计特征并归一化
    """
    curve = curve_144[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    
    # 填充 NaN
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any(): 
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)
    
    # 全局卡尔曼滤波 (在完整序列上滤波效果最好)
    curve = apply_kalman_filter(curve)

    # 生成切片 (Sliding Windows)
    slices = []
    if is_train:
        # 训练集：滑动截取 (默认生成 7 个切片)
        for offset in range(0, 144 - window_size + 1, stride):
            slices.append(curve[offset : offset + window_size])
    else:
        # 验证/测试集：只取最正中的一段，保证评估一致性
        center_offset = (144 - window_size) // 2
        slices.append(curve[center_offset : center_offset + window_size])

    results = []
    for sl in slices:
        min_val, max_val = sl.min(), sl.max()
        denom = max_val - min_val + 1e-8
        norm_curve = (sl - min_val) / denom
        
        mean_val, std_val = np.mean(norm_curve), np.std(norm_curve)
        log_max_val = np.log1p(max_val) / 10.0 
        stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

        grad_curve = np.gradient(norm_curve)
        two_channel_seq = np.stack([norm_curve, grad_curve], axis=0).astype(np.float32)
        results.append((two_channel_seq, stats))
        
    return results

class PEMS_SlidingWindowDataset(Dataset):
    def __init__(self, split_path, labels, allowed_days=None, is_train=False, window_size=132, stride=2, split_name=""):
        self.samples = []
        
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            # 如果提供了 allowed_days，只读取允许的天数（用于严格区分 Train 和 Val）
            if allowed_days is not None and day_i not in allowed_days:
                continue
                
            if mat is None or mat.shape[0] != TOTAL_STATIONS or mat.shape[1] < 144: 
                continue
                
            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            for local_sidx in range(TOTAL_STATIONS):
                # 调用滑动窗口处理逻辑
                processed_slices = process_and_slice_curve(mat[local_sidx, :144], is_train, window_size, stride)
                
                for seq, stats in processed_slices:
                    if not np.isfinite(seq).all(): continue
                    meta = {
                        "station_pos": int(local_sidx),
                        "day_i": int(day_i), 
                        "split": split_name,
                    }
                    self.samples.append((seq, stats, y, meta))
                
        self.n = len(self.samples)
        print(f"[{split_name}] 加载完成: 包含 {self.n} 条样本 (is_train={is_train}).")

    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        
        # 依然保留轻微的数值扰动增强
        if meta['split'] == 'train_augmented':
            seq_tensor = seq_tensor * random.uniform(0.95, 1.05)
            seq_tensor += torch.randn_like(seq_tensor) * 0.005
            
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

In [ ]:
# ==========================================
# Cell 4: CNN-BiLSTM 架构 (自动适配 132 长度)
# ==========================================
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout1d(0.1)
        )
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.2
        )
        # 全局池化完美解决了长度变化的问题
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), 
            nn.Linear(128 + 3, 64), 
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        cnn_out = self.cnn(x) 
        lstm_in = cnn_out.permute(0, 2, 1) 
        lstm_out, _ = self.lstm(lstm_in) 
        lstm_out_pool = lstm_out.permute(0, 2, 1) 
        features = self.gap(lstm_out_pool).view(lstm_out_pool.size(0), -1) 
        return self.fc(torch.cat([features, stats], dim=1))

In [ ]:
# ==========================================
# Cell 5: 按“天”严格划分并执行训练
# ==========================================
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'

# 1. 严格按天数 (Days) 划分训练集和验证集，杜绝数据穿越
total_days = len(labels_train)
all_days = torch.randperm(total_days, generator=torch.Generator().manual_seed(42)).tolist()
train_days_set = set(all_days[:int(total_days * 0.8)])
val_days_set   = set(all_days[int(total_days * 0.8):])

print("\n>>> 开始生成滑动窗口数据集 (这一步会产生海量数据，请稍候)...")
# Train: 开启滑动增强 (is_train=True)
ds_train = PEMS_SlidingWindowDataset(train_path, labels_train, allowed_days=train_days_set, is_train=True, split_name="train_augmented")
# Val: 关闭滑动增强，仅取中心窗口 (is_train=False)
ds_val   = PEMS_SlidingWindowDataset(train_path, labels_train, allowed_days=val_days_set, is_train=False, split_name="val_clean")
# Test: 关闭滑动增强，仅取中心窗口
ds_test  = PEMS_SlidingWindowDataset(test_path, labels_test, allowed_days=None, is_train=False, split_name="test")

loaders = {
    'train': DataLoader(ds_train, batch_size=512, shuffle=True, num_workers=0), # Batch size 调大以加速
    'val':   DataLoader(ds_val, batch_size=512, shuffle=False, num_workers=0),
    'test':  DataLoader(ds_test, batch_size=512, shuffle=False, num_workers=0),
}

def train_sliding_window_model(loaders, epochs=50):
    model = CNN_BiLSTM_Model().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc, best_state, counter, patience = 0.0, None, 0, 10
    
    print("\n>>> 开始训练大一统模型 (V3.2.0 - Sliding Window Augmented)...")
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _ in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x, stats), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step() 
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y, _ in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                correct += (torch.argmax(model(x, stats), dim=1) == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        if val_acc > best_acc:
            best_acc, best_state, counter = val_acc, model.state_dict(), 0
        else:
            counter += 1
            
        print(f"[Epoch {epoch}/{epochs}] Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.6f}")
        
        if counter >= patience:
            print(f"早停触发于 epoch {epoch}")
            break
            
    return best_state, best_acc

global_model_state, final_acc = train_sliding_window_model(loaders, epochs=50)
print(f"\n模型训练完成！最佳验证准确率: {final_acc:.2%}")
if global_model_state:
    torch.save(global_model_state, MODEL_DIR / 'cnn_lstm_v320_sliding.pth')

In [ ]:
# ==========================================
# Cell 6: 可视化与测试集结果导出
# ==========================================
def evaluate_model(model_state, loader, mode='test'):
    model = CNN_BiLSTM_Model().to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    y_true, y_pred, metas = [], [], []
    with torch.no_grad():
        for x, stats, y, meta in loader:
            x, stats = x.to(DEVICE), stats.to(DEVICE)
            preds = torch.argmax(model(x, stats), dim=1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            if isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    metas.append({k: meta[k][i] for k in meta})

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
    
    plt.figure(figsize=(10, 8))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f"V3.2.0 Sliding Window Augmented ({mode.capitalize()} Set)", fontsize=16, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    save_path = CM_DIR / f'matrix_{mode}_v320.png'
    plt.savefig(save_path, dpi=300)
    print(f"\n{mode.capitalize()} 集混淆矩阵已保存: {save_path}")
    plt.close()
    
    rows = []
    for i in range(len(y_true)):
        m = metas[i]
        yt, yp = y_true[i], y_pred[i]
        rows.append({
            "station_pos": int(m.get("station_pos", -1)),
            "day_i": int(m.get("day_i", -1)), 
            "y_true": int(yt), "y_pred": int(yp),
            "true_label": labels[yt], "pred_label": labels[yp],
            "correct": bool(yt == yp),
        })

    df = pd.DataFrame(rows).sort_values(["correct","station_pos","day_i"]).reset_index(drop=True)
    out_all = CM_DIR / f"predictions_{mode}_v320.csv"
    df.to_csv(out_all, index=False, encoding="utf-8-sig")
    print(f"🏆 {mode.capitalize()} 集最终准确率: {df['correct'].mean():.2%}")

evaluate_model(global_model_state, loaders['val'], 'val')
evaluate_model(global_model_state, loaders['test'], 'test')